# Retail Sales Performance Analysis

Dataset: Superstore Dataset
Source: https://www.kaggle.com/datasets/vivek468/superstore-dataset-final

Objectives:

- Analyze sales performance, profitability, and trends.
- Identify top categories/sub-categories and regional performance.
- Track growth over time and seasonality.

Tools:

- pandas
- matplotlib


## Data Loading


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from matplotlib.ticker import FuncFormatter

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("../data/Sample - Superstore.csv", encoding="latin1")

## Data Cleaning


Look at structure

Things to check:

- column names
- strange formatting
- unexpected values


In [ ]:
df.head()

Check data types


In [ ]:
df.info()

In [ ]:
# Convert date values to datetime
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

Check missing values


In [ ]:
df.isna().sum()

Check duplicates

Duplicate records can inflate results


In [ ]:
df.duplicated().sum()

Check value ranges

Look for suspicious values


In [ ]:
df.describe().T

This dataset does not contain significant data quality issues.

Therefore, the cleaning process mainly consisted of reviewing the data structure and ensuring that all variables have the appropriate data types to facilitate analysis.


## Exploratory Data Analysis (EDA)


Some questions we can answer

Sales and Profit:

- Which categories have the best sales
- Which categories give the most profits

Customer:

- Which customer segment gives the most profit
- What makes a customer buy something again in the store (discounts, product categories, delivery time, etc.)

Time:

- When can we see the most sales and profits

Discounts:

- Are the discounts improve profits?
- Are the discounts afecting future sales in some product?
- Is there a time to time to give discounts (day, week, month)


#### Sales vs Profit by Category


In [ ]:
# sales and profit by category
category_summary = df.groupby("Category")[["Sales", "Profit"]].sum()

# create the plot view
fig, ax = plt.subplots(figsize=(10, 6))
category_summary.plot(kind="bar", ax=ax)

# setting labels
ax.set_title("Sales vs Profit by Category")
ax.set_ylabel("Amount")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x/1_000:,.0f}K'))

# setting grid
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

# add more space at the top of the grapth to display texts above the bars
max_value = category_summary.max().max()
ax.set_ylim(0, max_value * 1.15)

# calculate the totals for sales and profit
total_sales = category_summary["Sales"].sum()
total_profit = category_summary["Profit"].sum()

# add totals and share values above each bar
for container, metric in zip(ax.containers, ["Sales", "Profit"]):
    total = total_sales if metric == "Sales" else total_profit

    for bar in container:
        height = bar.get_height()
        percent = height / total * 100

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"${height:,.0f}\n({percent:.1f}%)",
            ha="center",
            va="bottom",
            color="black",
            fontsize=9
        )

plt.show()

- Technology represents the most profit (~51% of the total profits)
- Office Supplies is the second most profitable category
- Furniture represents a very little profit compared to its sales. Only ~6.4% of the total profit compared to ~32% of the total sales


#### Sales vs Profit by Customer Segment


In [ ]:
# customer segment analysis
customer_segment_summary = df.groupby("Segment")[["Sales", "Profit"]].sum()

# create the plot view
fig, ax = plt.subplots(figsize=(10, 6))
customer_segment_summary.plot(kind="bar", ax=ax)

# setting labels
ax.set_title("Sales vs Profit by Customer Segment")
ax.set_ylabel("Amount")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x/1_000_000:.1f}M'))

# setting grid
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

# add more space at the top of the grapth to display texts above the bars
max_value = customer_segment_summary.max().max()
ax.set_ylim(0, max_value*1.15)

# calculate the totals for sales and profit
total_sales = customer_segment_summary["Sales"].sum()
total_profit = customer_segment_summary["Profit"].sum()

# add totals and share values above each bar
for container, metric in zip(ax.containers, ["Sales", "Profit"]):
    total = total_sales if metric == "Sales" else total_profit

    for bar in container:
        height = bar.get_height()
        percent = height / total * 100

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"${height:,.0f}\n({percent:.1f}%)",
            ha="center",
            va="bottom",
            color="black",
            fontsize=9
        )

plt.show()

## ...


Identify Outliers


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
df.plot(column="Sales", by="Category", kind="box", ax=ax)

ax.set_yscale("log")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.grid(axis="y", linestyle="--", alpha=0.7)

ax.set_title("Sales Distribution by Category")
ax.set_ylabel("Sales")

plt.show()

- Technology and Furniture have higher median sales
- Technology has the most extreme high-value transactions
- Office Supplies has lower-value


In [ ]:
df.describe(include='all').T.sort_values("unique")

In [ ]:
# totals
total_sales = df["Sales"].sum()
total_per_year = df.groupby(df["Order Date"].dt.to_period("Y"))["Sales"].sum().reset_index()

# calculate growth over years
total_per_year["Growth"] = total_per_year["Sales"].diff().fillna(0)
total_per_year["Percent Growth"] = total_per_year["Sales"].pct_change().fillna(0) * 100

total_per_year

In [ ]:
# plot sales by time
sales_by_time = df.groupby(df["Order Date"].dt.to_period("M"))["Sales"].sum()

rolling_mean_12 = sales_by_time.rolling(12).mean()
sales_by_time.plot(label="Monthly Sales")
rolling_mean_12.plot(label="12-Month Average")

plt.legend()

In [ ]:
# average sales by month of the year
avg_monthly_sales = (
    df.groupby(df["Order Date"].dt.to_period("M"))["Sales"]
    .sum()
    .groupby(lambda x: x.month)
    .mean()
)
# correct indexes
avg_monthly_sales.index = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

avg_monthly_sales.plot(kind="bar")

In [ ]:
# sales by category
sales_by_category = df.groupby("Category")["Sales"].sum().sort_values(ascending=False).reset_index()
sales_by_category["Share"] = sales_by_category["Sales"] / df["Sales"].sum() * 100

ax = sales_by_category.plot(
    kind="barh",
    x="Category",
    y="Share",
    legend=False
)

bars = ax.barh(
    sales_by_category["Category"],
    sales_by_category["Share"]
)

for bar, sales in zip(bars, sales_by_category["Sales"]):
    width = bar.get_width()
    ax.text(
        width + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}% (${sales:,.0f})",
        va="center"
    )

plt.xlabel("Sales Share (%)")
plt.title("Sales Share by Category")
plt.xlim(0, 100)
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# sales by region
sales_by_region = df.groupby("Region")["Sales"].sum().reset_index()
sales_by_region["Share"] = sales_by_region["Sales"] / df["Sales"].sum() * 100

ax = sales_by_region.plot(
    kind="barh",
    x="Region",
    y="Share",
    legend=False
)

bars = ax.barh(
    sales_by_region["Region"],
    sales_by_region["Share"]
)

for bar, sales in zip(bars, sales_by_region["Sales"]):
    width = bar.get_width()
    ax.text(
        width + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}% (${sales:,.0f})",
        va="center"
    )

plt.xlabel("Sales Share (%)")
plt.title("Sales Share by Region")
plt.xlim(0, 100)
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# sales by customer segment
sales_by_customer_segment = df.groupby("Segment")["Sales"].sum().sort_values(ascending=False).reset_index()
sales_by_customer_segment["Share"] = sales_by_customer_segment["Sales"] / df["Sales"].sum() * 100

ax = sales_by_customer_segment.plot(
    kind="barh",
    x="Segment",
    y="Share",
    legend=False
)

bars = ax.barh(
    sales_by_customer_segment["Segment"],
    sales_by_customer_segment["Share"]
)

for bar, sales in zip(bars, sales_by_customer_segment["Sales"]):
    width = bar.get_width()
    ax.text(
        width + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}% (${sales:,.0f})",
        va="center"
    )

plt.xlabel("Sales Share (%)")
plt.title("Sales Share by Customer Segment")
plt.xlim(0, 100)
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# top 10 products by revenue
top_revenue_products = (
    df.groupby(["Product Name", "Category", "Sub-Category"])["Sales"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .head(10)
)
top_revenue_products["Share"] = top_revenue_products["Sales"] / df["Sales"].sum() * 100

top_revenue_products

In [ ]:
# top 10 sub-categories
top_revenue_subcategories = (
    df.groupby("Sub-Category")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .head(10)
)

top_revenue_subcategories["Share"] = top_revenue_subcategories["Sales"] / df["Sales"].sum() * 100

top_revenue_subcategories

In [ ]:
# revenue by weekday
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
revenue_by_weekday = (
    df.groupby(df["Order Date"].dt.day_name())["Sales"]
    .sum()
    .reindex(weekday_order)
)

revenue_by_weekday.plot(kind="bar", title="Revenue by Weekday")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.show()